## Introduction
The goal of this project is to perform a Recency Frequency Monetary (RFM) analysis on transactional retail data from an online retailer to identify high-value customers to produce business insights. 

## Analysis

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('/Users/sumerafaisal/BI_proj_1/retail.db')

query = """
SELECT 
    CustomerID,
    julianday('2011-12-10') - julianday(MAX(InvoiceDate)) AS recency,
    COUNT(DISTINCT InvoiceNo) AS frequency,
    SUM(TotalPrice) AS monetary
FROM transactions
GROUP BY CustomerID
"""


After loading the data into SQL, I created a table containing CustomerID, recency, frequency, and monetary. Recency was calculated by taking the end date of the data collection as the "current date" and finding the time between then and a customers most recent transaction. This decision was made to avoid inflating numbers due to the age of this dataset (***NOTE FOR EDIT: not sure if that's actually true but I feel like I should explain). 
Frequency was defined by the number of distinct invoice numbers each customer made, as the dataset contains an entry per product in a transaction, so they needed to be defined per customer transaction. Finally, monetary was extracted through summing all the total prices of each product a customer purchased.

In [5]:
print(rfm_df.describe())

         CustomerID      recency    frequency       monetary
count   4338.000000  4338.000000  4338.000000    4338.000000
mean   15300.408022    92.514594     4.272015    2054.266460
std     1721.808492   100.013298     7.697998    8989.230441
min    12346.000000     0.465278     1.000000       3.750000
25%    13813.250000    17.537674     1.000000     307.415000
50%    15299.500000    50.555208     2.000000     674.485000
75%    16778.750000   142.195833     5.000000    1661.740000
max    18287.000000   373.588194   209.000000  280206.020000


A small subset of customers exhibits significantly higher spending and purchase frequency, indicating bulk buyers, as was mentioned in the dataset. Additionally, the frequency statistics report the median frequency of purchases at 2, whereas the max number of transactions is 209, indicating that customer engagement is highly uneven, with a small group of repeat buyers driving consistent activity. The skew observed in the data indicates the potential need for grouping.

In [ ]:
rfm_df = pd.read_sql(query, conn)
print("Top by Monetary:")
print(rfm_df.sort_values(by='monetary', ascending=False).head())

print("\nTop by Recency:")
print(rfm_df.sort_values(by='recency', ascending=True).head())

print("\nTop by Frequency:")
print(rfm_df.sort_values(by='frequency', ascending=False).head())

Top by Monetary:
      CustomerID   recency  frequency   monetary
1689     14646.0  1.491667         73  280206.02
4201     18102.0  0.506944         60  259657.30
3728     17450.0  8.438194         46  194550.79
3008     16446.0  0.614583          2  168472.50
1879     14911.0  1.337500        201  143825.06

Top by Recency:
      CustomerID   recency  frequency  monetary
271      12680.0  0.465278          4    862.81
581      13113.0  0.465972         24  12245.96
2542     15804.0  0.478472         13   4206.39
1058     13777.0  0.482639         33  25977.16
3823     17581.0  0.485417         25  11045.04

Top by Frequency:
      CustomerID   recency  frequency   monetary
326      12748.0  0.486111        209   33719.73
1879     14911.0  1.337500        201  143825.06
4010     17841.0  1.495139        124   40991.57
562      13089.0  2.623611         97   58825.83
1661     14606.0  1.188889         93   12156.65


## NOTE FOR EDIT not sure if I need to include things like this where I was just getting familiar with the stats within the data

## RFM Scoring

In [6]:
rfm_df['R_score'] = pd.qcut(rfm_df['recency'], 5, labels=[5,4,3,2,1])
rfm_df['F_score'] = pd.qcut(rfm_df['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm_df['M_score'] = pd.qcut(rfm_df['monetary'], 5, labels=[1,2,3,4,5])


I created quintiles for each RFM category, with 5 being a great score, and 1 being weak. The labelling is reversed for recency because 5's go to customers with the lowest numbers, as that means they have purchased more recently. For both the frequency and monetary categories, the value assignment is reversed, as we want the highest values to receive 5's. 

In [7]:
rfm_df['RFM_score'] = (
    rfm_df['R_score'].astype(int) +
    rfm_df['F_score'].astype(int) +
    rfm_df['M_score'].astype(int)
)

Adding scores of each component to get the final RFM score of each customer. Scores will range from 3, being the lowest value customers, to 15, at highest value. 

## Customer Segmentation

In [9]:
def segment_customer(score):
    if score >= 12:
        return 'High Value'
    elif score >= 8:
        return 'Mid Value'
    else:
        return 'Low Value'

rfm_df['segment'] = rfm_df['RFM_score'].apply(segment_customer)

Given the disparity between bulk-buyers and regular customers noted earlier, I decided to segment customers into low, mid, and high value groups based on their RFM score.

In [10]:
rfm_df.groupby('segment')['monetary'].sum()

segment
High Value    6812949.950
Low Value      637225.851
Mid Value     1461232.103
Name: monetary, dtype: float64

NOTE FOR EDIT: How can I get the values in the table in order from low to high value customers?
This reveals that low value customers generate 7%, mid value customers produce 16%, and high-value customers generate 77% of the company's monetary income. This is likely due to their presence as an exclusive online store. 

In [11]:
rfm_df['segment'].value_counts(normalize=True)

segment
Low Value     0.390272
Mid Value     0.322499
High Value    0.287229
Name: proportion, dtype: float64

NOTE FOR EDITS: not really sure why we need to calculate segment sizes. Aren't they naturally going to be split up roughly evenly given the format of the RFM analysis?

In [13]:
rfm_df.groupby('segment')[['recency','frequency','monetary']].mean()

,recency,frequency,monetary
segment,,,
High Value,18.967304,9.988764,5467.857103
Low Value,170.318072,1.239220,376.388571
Mid Value,63.864584,2.850608,1044.483276


Taking the average of each customer segment's RFM score shows that mid-value customers have the potential to bring more EDIT: not really sure what to say here. My general understanding is you would want to target the mid-value customers because they still have relatively high monetary contributions, but their recency and frequency numbers are moderate.